# **🛫 항공편 지연 예측 모델 고도화 파이프라인**

**핵심 전략: Cyclic Encoding, Class Weighting, Threshold Optimization, Pseudo-Labeling**

**코드 환경 변환 (py -> ipynb)**

In [1]:
# 구글 드라이브 연결
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/AI/Data Set/항공편 지연 예측

Mounted at /content/drive
/content/drive/MyDrive/AI/Data Set/항공편 지연 예측


In [3]:
# 라이브 설치
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.6 MB/s eta 0:00:00


# **1. 환경 설정 및 데이터 로드<br>**
**파일명 참조: 피처 엔지니어링(시간최적화)_1단계.py**<br>
**대용량 데이터의 효율적인 처리를 위해 메모리를 최적화하여 로드합니다.**

In [7]:
import pandas as pd
import numpy as np
from catboost import CatBoostClassifier
from sklearn.metrics import precision_recall_curve, f1_score, classification_report
import matplotlib.pyplot as plt

# 메모리 최적화 함수
def safe_reduce_mem_usage(df):
    numeric_cols = df.select_dtypes(include=['number']).columns
    for col in numeric_cols:
        c_min, c_max = df[col].min(), df[col].max()
        if pd.api.types.is_integer_dtype(df[col]):
            if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                df[col] = df[col].astype(np.int8)
            else:
                df[col] = df[col].astype(np.int32)
        else:
            df[col] = df[col].astype(np.float32)
    return df

# 데이터 로드
train = pd.read_csv('dataset/train.csv')
train = safe_reduce_mem_usage(train)

# **2. 피처 엔지니어링 (시간 데이터 최적화)<br>**
**파일명 참조: 피처 엔지니어링(시간최적화)_1단계.py, 모델 학습 및 검증.py**

**시간의 주기성(23시와 00시의 인접성)을 반영하기 위해 Sin/Cos 변환을 적용합니다.**

In [8]:
def process_time_features(df):
    for col in ['Estimated_Departure_Time', 'Estimated_Arrival_Time']:
        df[col] = df[col].fillna(-1)
        df[f'{col}_Hour'] = df[col].apply(lambda x: x // 100 if x != -1 else -1)

        # Cyclic Encoding
        df[f'{col}_Hour_Sin'] = np.where(df[f'{col}_Hour'] != -1,
                                         np.sin(2 * np.pi * df[f'{col}_Hour'] / 24), -1)
        df[f'{col}_Hour_Cos'] = np.where(df[f'{col}_Hour'] != -1,
                                         np.cos(2 * np.pi * df[f'{col}_Hour'] / 24), -1)
    return df

train = process_time_features(train)
# 타겟 라벨링 (Delayed: 1, Not_Delayed: 0)
label_map = {'Not_Delayed': 0, 'Delayed': 1, 0: 0, 1: 1}
train['Delay_Numeric'] = train['Delay'].map(label_map)

# **3. 클래스 불균형 해소 및 베이스라인 모델 학습**
**파일명 참조: 모델 학습 및 검증.py**

**정상 대비 지연 데이터의 부족 문제를 해결하기 위해 가중치(scale_pos_weight)를 적용합니다.**

In [9]:
# 정답이 있는 데이터만 추출
labeled_df = train.dropna(subset=['Delay_Numeric']).copy()

# 가중치 계산
num_neg = len(labeled_df[labeled_df['Delay_Numeric'] == 0])
num_pos = len(labeled_df[labeled_df['Delay_Numeric'] == 1])
scale_weight = num_neg / num_pos

# 모델 학습
features = ['Estimated_Departure_Time_Hour_Sin', 'Estimated_Departure_Time_Hour_Cos'] # 실제 사용 피처
model = CatBoostClassifier(iterations=1000, scale_pos_weight=scale_weight, random_seed=42, verbose=100)
model.fit(labeled_df[features], labeled_df['Delay_Numeric'].astype(int))

Learning rate set to 0.109781
0:	learn: 0.6894199	total: 109ms	remaining: 1m 48s
100:	learn: 0.6751057	total: 7.68s	remaining: 1m 8s
200:	learn: 0.6750463	total: 13s	remaining: 51.5s
300:	learn: 0.6750766	total: 21s	remaining: 48.8s
400:	learn: 0.6750790	total: 27.6s	remaining: 41.2s
500:	learn: 0.6750916	total: 32.6s	remaining: 32.5s
600:	learn: 0.6750939	total: 41.4s	remaining: 27.5s
700:	learn: 0.6751043	total: 46.4s	remaining: 19.8s
800:	learn: 0.6751027	total: 53.8s	remaining: 13.4s
900:	learn: 0.6751027	total: 59.2s	remaining: 6.5s
999:	learn: 0.6751019	total: 1m 4s	remaining: 0us


CatBoostClassifier(iterations=1000, random_seed=42, scale_pos_weight=4.666688888888889, verbose=100)

# **4. 임계값 최적화 (Threshold Tuning)**
**파일명 참조: 임계값 최적화 코드.py, 현재 모델 상세 평가.py**

**F1-Score를 극대화하는 최적의 임계값(0.4866)을 도출합니다.**

In [10]:
y_prob = model.predict_proba(labeled_df[features])[:, 1]
precisions, recalls, thresholds = precision_recall_curve(labeled_df['Delay_Numeric'], y_prob)
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-9)

best_threshold = thresholds[np.argmax(f1_scores)]
print(f"최적 임계값: {best_threshold:.4f}")

최적 임계값: 0.4866


# **5. Pseudo-Labeling (준지도 학습 고도화)**
**파일명 참조: 최종 모델 완성.py**

**정답이 없는 75%의 결측치 데이터를 활용하여 모델의 성능을 한 단계 더 끌어올립니다.**

In [11]:
unlabeled_df = train[train['Delay'].isna()].copy()
probs = model.predict_proba(unlabeled_df[features])[:, 1]

# 확신도 90% 이상인 데이터에 라벨 부여
unlabeled_df['Delay_Numeric'] = -1
unlabeled_df.loc[probs >= 0.90, 'Delay_Numeric'] = 1
unlabeled_df.loc[probs <= 0.10, 'Delay_Numeric'] = 0

pseudo_labeled = unlabeled_df[unlabeled_df['Delay_Numeric'] != -1].copy()
final_train_df = pd.concat([labeled_df, pseudo_labeled], axis=0)

# 최종 ULTIMATE 모델 학습
ultimate_model = CatBoostClassifier(iterations=1000, scale_pos_weight=scale_weight, random_seed=42, verbose=100)
ultimate_model.fit(final_train_df[features], final_train_df['Delay_Numeric'].astype(int))

Learning rate set to 0.109781
0:	learn: 0.6894199	total: 129ms	remaining: 2m 8s
100:	learn: 0.6751057	total: 7.68s	remaining: 1m 8s
200:	learn: 0.6750463	total: 12.6s	remaining: 50s
300:	learn: 0.6750766	total: 20.5s	remaining: 47.7s
400:	learn: 0.6750790	total: 28.7s	remaining: 42.8s
500:	learn: 0.6750916	total: 36.7s	remaining: 36.6s
600:	learn: 0.6750939	total: 41.8s	remaining: 27.8s
700:	learn: 0.6751043	total: 49.2s	remaining: 21s
800:	learn: 0.6751027	total: 55.4s	remaining: 13.8s
900:	learn: 0.6751027	total: 1m	remaining: 6.63s
999:	learn: 0.6751019	total: 1m 8s	remaining: 0us


CatBoostClassifier(iterations=1000, random_seed=42, scale_pos_weight=4.666688888888889, verbose=100)